In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('') #Add your dir

In [ ]:
import torch
# If there's a GPU available...
if torch.cuda.is_available():

    # Tell PyTorch to use the GPU.
    device = torch.device("cuda")

    print('There are %d GPU(s) available.' % torch.cuda.device_count())

    print('We will use the GPU:', torch.cuda.get_device_name(0))
    !nvidia-smi

# If not...
else:
    print('No GPU available, using the CPU instead.')
    device = torch.device("cpu")


# 1. load required models, configurations, data, and Convert data to Huggingface format.


In [ ]:
# =====================================================
# Install Required Packages
# =====================================================
!pip install -q transformers datasets accelerate peft bitsandbytes
!pip install -q scikit-learn pandas numpy matplotlib seaborn
!pip install -q torch torchvision torchaudio

print("Installation complete!")

In [ ]:
# =====================================================
# Import Libraries
# =====================================================
import torch
import pandas as pd
import numpy as np
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)
import matplotlib.pyplot as plt
import seaborn as sns
import json
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Check GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# =====================================================
# Configuration
# =====================================================

DIALECTS = {0: "Hijazi", 1: "Janobi", 2: "Najdi", 3: "Hasawi"}
LABEL2ID = {"Hijazi": 0, "Janobi": 1, "Najdi": 2, "Hasawi": 3}
ID2LABEL = {0: "Hijazi", 1: "Janobi", 2: "Najdi", 3: "Hasawi"}
NUM_LABELS = 4

SORTED_LABEL_IDS = sorted(ID2LABEL)                            # [0, 1, 2, 3]
SORTED_LABEL_NAMES = [ID2LABEL[i] for i in SORTED_LABEL_IDS]   # names in ID order

FINETUNING_MODELS = {
    'marbert': 'UBC-NLP/MARBERT',
    'arabert-v2': 'aubmindlab/bert-base-arabertv2',
    'camelbert-da': 'CAMeL-Lab/bert-base-arabic-camelbert-da',
    'allam-7b': 'ALLaM-AI/ALLaM-7B-Instruct-preview',
    'qwen2.5-7b': 'Qwen/Qwen2.5-7B-Instruct',
    'qarib':'ahmedabdelali/bert-base-qarib',
}
ZEROSHOT_MODELS = {
    'qwen2.5-7b': 'Qwen/Qwen2.5-7B-Instruct',
    'allam-7b': 'ALLaM-AI/ALLaM-7B-Instruct-preview',
    'mistral-7b': 'mistralai/Mistral-7B-Instruct-v0.3',
}
# ============================================================
# SETTINGS - SAME FOR ALL MODELS
# ============================================================
NUM_EPOCHS = 5
LEARNING_RATE = 2e-5
MAX_LENGTH = 128
SEEDS = [42, 123, 2024]


# Model-specific settings
MODEL_CONFIGS = {
    # Small models
    'marbert': {
        'batch_size': 16,
        'gradient_accumulation_steps': 1,
        'use_lora': False,
        'use_8bit': False,
    },
    'arabert-v2': {
        'batch_size': 16,
        'gradient_accumulation_steps': 1,
        'use_lora': False,
        'use_8bit': False,
    },
    'camelbert-da': {
        'batch_size': 16,
        'gradient_accumulation_steps': 1,
        'use_lora': False,
        'use_8bit': False,
    },
    'qarib': {
        'batch_size': 16,
        'gradient_accumulation_steps': 1,
        'use_lora': False,
        'use_8bit': False,
    },

    # Large models - batch size set to 4 to keep effective batch similar = 16
    'allam-7b': {
        'batch_size': 4,
        'gradient_accumulation_steps': 4,
        'use_lora': True,
        'use_8bit': True,
        'lora_r': 16,
        'lora_alpha': 32,
    },

    'qwen2.5-7b': {
        'batch_size': 4,
        'gradient_accumulation_steps': 4,
        'use_lora': True,
        'use_8bit': True,
        'lora_r': 16,
        'lora_alpha': 32,
    },
}

# LoRA settings
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.1

print("Fair comparison configuration loaded")
print(f"  Epochs (all models): {NUM_EPOCHS}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Effective batch size: ~16 for all models")
print(f"  Dialect ID mapping : {ID2LABEL}")
print(f"  Seeds for multi-seed runs: {SEEDS}")


In [ ]:
# =====================================================
# Uplpoad Training, Test and Validation set
# =====================================================

train_df=pd.read_csv("train_dialectdf.csv")
val_df=pd.read_csv("val_dialectdf.csv")
test_df=pd.read_csv("test_dialectdf.csv")

In [ ]:
# =====================================================
# Convert to HuggingFace Dataset Format
# =====================================================

dataset_dict = DatasetDict({
    'train': Dataset.from_pandas(train_df[['text', 'label']]),
    'validation': Dataset.from_pandas(val_df[['text', 'label']]),
    'test': Dataset.from_pandas(test_df[['text', 'label']])
})

print("\n Dataset created successfully!")
print(f"\nSample from training set:")
print(dataset_dict['train'][0])

#2. Fine-tuned Experiment

##2.1 Fine-Tuned Functions

In [ ]:
# =====================================================
# Helper Functions
# =====================================================

def set_seed(seed=2024):
    """Set random seed for reproducibility"""
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)

    precision_w, recall_w, f1_w, _ = precision_recall_fscore_support(
        labels, predictions, average='weighted', zero_division=0
    )
    precision_m, recall_m, f1_m, _ = precision_recall_fscore_support(
        labels, predictions, average='macro', zero_division=0
    )

    return {
        'accuracy': accuracy,
        'precision': precision_m,
        'recall': recall_m,
        'f1': f1_m,                     # used by early stopping
        'precision_macro': precision_m,
        'recall_macro': recall_m,
        'f1_macro': f1_m,
        'precision_weighted': precision_w,
        'recall_weighted': recall_w,
        'f1_weighted': f1_w,
    }

def plot_confusion_matrix(cm, title='Confusion Matrix'):

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=SORTED_LABEL_NAMES,
                yticklabels=SORTED_LABEL_NAMES)
    plt.title(title)
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()

def print_classification_report(labels, predictions, model_name):
    """Print detailed classification report"""
    print(f"\n{'='*60}")
    print(f"CLASSIFICATION REPORT: {model_name}")
    print(f"{'='*60}")
    print(classification_report(
        labels,
        predictions,
        labels=SORTED_LABEL_IDS,
        target_names=SORTED_LABEL_NAMES,
        digits=4
    ))
    print(f"{'='*60}\n")

# =====================================================
# Significance-testing function
# =====================================================

def mcnemar_test(true_labels, pred_a, pred_b, model_a_name="Model A", model_b_name="Model B"):
    """
    Paired McNemar's test between two models' predictions on the SAME
    test items, in the SAME order. Requires statsmodels:
        !pip install -q statsmodels
    """
    from statsmodels.stats.contingency_tables import mcnemar as _mcnemar

    true_labels = np.asarray(true_labels)
    pred_a = np.asarray(pred_a)
    pred_b = np.asarray(pred_b)
    assert len(true_labels) == len(pred_a) == len(pred_b), \
        "true_labels, pred_a, and pred_b must be the same length and item order"

    correct_a = (pred_a == true_labels)
    correct_b = (pred_b == true_labels)

    both_correct = int(np.sum(correct_a & correct_b))
    a_only = int(np.sum(correct_a & ~correct_b))
    b_only = int(np.sum(~correct_a & correct_b))
    both_wrong = int(np.sum(~correct_a & ~correct_b))

    table = [[both_correct, a_only], [b_only, both_wrong]]
    use_exact = min(a_only, b_only) < 25
    result = _mcnemar(table, exact=use_exact, correction=True)

    print(f"McNemar's test: {model_a_name} vs {model_b_name}")
    print(f"  {model_a_name} correct only: {a_only}   {model_b_name} correct only: {b_only}")
    print(f"  statistic = {result.statistic:.4f}, p-value = {result.pvalue:.4f}"
          f"  ({'exact' if use_exact else 'chi-square'} test)")
    verdict = "=> significant difference (p<0.05)" if result.pvalue < 0.05 else "=> NOT significant at p<0.05"
    print(f"  {verdict}")
    return result.pvalue, table


def wilson_ci(n_correct, n_total, confidence=0.95):
    """95% Wilson confidence interval for an accuracy proportion."""
    from statsmodels.stats.proportion import proportion_confint
    low, high = proportion_confint(n_correct, n_total, alpha=1 - confidence, method='wilson')
    point = n_correct / n_total
    return point, low, high


def print_wilson_ci_table(model_results):
    """
    model_results: dict of {model_name: (n_correct, n_total)}
    Prints a table of accuracy with 95% Wilson confidence intervals.
    """
    print(f"{'Model':<20} {'Accuracy':<12} {'95% Wilson CI':<20}")
    print("-" * 52)
    for name, (n_correct, n_total) in model_results.items():
        point, low, high = wilson_ci(n_correct, n_total)
        print(f"{name:<20} {point*100:>6.2f}%      [{low*100:.2f}%, {high*100:.2f}%]")

print("Helper functions loaded (macro-F1 fix, explicit label ordering, significance-testing helpers)")


##2.2 Fine Tuning function model


In [ ]:
def finetune_model(model_name, model_id, dataset_dict, seed=None):
    """
    Fine-tune a single model.

    Args:
        model_name: Short name (e.g., 'allam-7b')
        model_id: HuggingFace model ID
        dataset_dict: Your dataset
        seed: Defaults to the global SEED (42)

    Returns:
        trainer, results dictionary
    """
    effective_seed = seed if seed is not None else SEED

    print(f"\n{'#'*60}")
    print(f"# Fine-tuning: {model_name}  (seed={effective_seed})")
    print(f"{'#'*60}\n")

    set_seed(effective_seed)

    if model_name in MODEL_CONFIGS:
        config = MODEL_CONFIGS[model_name]
        BATCH_SIZE = config['batch_size']
        GRAD_ACCUM = config.get('gradient_accumulation_steps', 1)
        USE_LORA = config['use_lora']
        USE_8BIT = config['use_8bit']
    else:
        BATCH_SIZE = 8
        GRAD_ACCUM = 2
        USE_LORA = False
        USE_8BIT = False

    print(f" Training config:")
    print(f"  Seed: {effective_seed}")
    print(f"  Batch size: {BATCH_SIZE}")
    print(f"  Gradient accumulation: {GRAD_ACCUM}")
    print(f"  Effective batch: {BATCH_SIZE * GRAD_ACCUM}")
    print(f"  Epochs: {NUM_EPOCHS}")
    print(f"  Learning rate: {LEARNING_RATE}")

    # 1. Load tokenizer
    print(f"Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # 2. Tokenize dataset
    def tokenize_function(examples):
        return tokenizer(
            examples['text'],
            padding='max_length',
            truncation=True,
            max_length=MAX_LENGTH
        )

    print("Tokenizing dataset...")
    tokenized_dataset = dataset_dict.map(
        tokenize_function,
        batched=True,
        remove_columns=['text']
    )

    # 3. Load model with 8-bit quantization
    print(f"Loading model: {model_id}")

    if USE_8BIT:
        bnb_config = BitsAndBytesConfig(
            load_in_8bit=True,
            bnb_8bit_compute_dtype=torch.float16,
            torch_dtype=torch.float16

        )

        model = AutoModelForSequenceClassification.from_pretrained(
            model_id,
            num_labels=NUM_LABELS,
            id2label=ID2LABEL,
            label2id=LABEL2ID,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True
        )

        model = prepare_model_for_kbit_training(model)
    else:
        model = AutoModelForSequenceClassification.from_pretrained(
            model_id,
            num_labels=NUM_LABELS,
            id2label=ID2LABEL,
            label2id=LABEL2ID,
            trust_remote_code=True
        )
        model.to(device)

    # 4. Apply LoRA
    if USE_LORA:
        print("Applying LoRA...")
        lora_config = LoraConfig(
            r=LORA_R,
            lora_alpha=LORA_ALPHA,
            target_modules=["q_proj", "v_proj"],
            lora_dropout=LORA_DROPOUT,
            bias="none",
            task_type=TaskType.SEQ_CLS
        )
        model = get_peft_model(model, lora_config)
        model.print_trainable_parameters()

    # 5. Training arguments
    training_args = TrainingArguments(
        output_dir=f"./results_{model_name}_seed{effective_seed}",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=NUM_EPOCHS,
        weight_decay=0.01,
        warmup_steps=100,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        save_total_limit=2,
        logging_steps="epoch",
        fp16=True,
        gradient_checkpointing=True if USE_LORA or USE_8BIT else False,
        report_to="none",
        seed=effective_seed,
    )

    # 6. Create trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset['train'],
        eval_dataset=tokenized_dataset['validation'],
        processing_class=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
    )

    # 7. Train
    print(f"\nStarting training...")
    model.gradient_checkpointing_enable()
    trainer.train()

    # ============================================================
    # SAVE THE TRAINED MODEL
    # ============================================================
    print(f"\nSaving trained model...")
    model_save_path = f"./saved_model_{model_name}_seed{effective_seed}"

    trainer.save_model(model_save_path)
    tokenizer.save_pretrained(model_save_path)

    config_info = {
        'model_name': model_name,
        'base_model_id': model_id,
        'num_labels': NUM_LABELS,
        'label2id': LABEL2ID,
        'id2label': ID2LABEL,
        'training_config': {
            'batch_size': BATCH_SIZE,
            'learning_rate': LEARNING_RATE,
            'num_epochs': NUM_EPOCHS,
            'max_length': MAX_LENGTH,
            'use_lora': USE_LORA,
            'use_8bit': USE_8BIT,
            'seed': effective_seed
        },
        'saved_date': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
    }

    with open(f"{model_save_path}/training_config.json", 'w') as f:
        json.dump(config_info, f, indent=2)

    print(f"Model saved to: {model_save_path}")

    import os
    saved_files = os.listdir(model_save_path)
    total_size = sum(os.path.getsize(os.path.join(model_save_path, f)) for f in saved_files) / (1024**2)
    print(f"Total size: {total_size:.1f} MB")
    print(f"Files saved: {len(saved_files)}")

    # 8. Evaluate on test set
    print(f"\nEvaluating on test set...")
    predictions = trainer.predict(tokenized_dataset['test'])

    pred_labels = np.argmax(predictions.predictions, axis=-1)
    true_labels = predictions.label_ids

    accuracy = accuracy_score(true_labels, pred_labels)
    precision_w, recall_w, f1_w, _ = precision_recall_fscore_support(
        true_labels, pred_labels, average='weighted', zero_division=0
    )
    precision_m, recall_m, f1_m, _ = precision_recall_fscore_support(
        true_labels, pred_labels, average='macro', zero_division=0
    )
    cm = confusion_matrix(true_labels, pred_labels, labels=SORTED_LABEL_IDS)

    # ============================================================
    # SAVE RESULTS TO TEXT FILE
    # ============================================================
    results_text = f"""
{'='*60}
RESULTS: {model_name}  (seed={effective_seed})
{'='*60}
Model ID: {model_id}
Training Date: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}
Model Saved At: {model_save_path}

OVERALL METRICS:
Accuracy:         {accuracy:.4f}
Macro Precision:  {precision_m:.4f}
Macro Recall:     {recall_m:.4f}
Macro F1:         {f1_m:.4f}   <-- standard metric
Weighted Precision: {precision_w:.4f}
Weighted Recall:    {recall_w:.4f}
Weighted F1:        {f1_w:.4f}
{'='*60}

DETAILED CLASSIFICATION REPORT:
{classification_report(true_labels, pred_labels, labels=SORTED_LABEL_IDS, target_names=SORTED_LABEL_NAMES, digits=4)}

CONFUSION MATRIX (rows/cols in order: {SORTED_LABEL_NAMES}):
{cm}

Per-class breakdown:
"""

    precision_per_class, recall_per_class, f1_per_class, support = \
        precision_recall_fscore_support(true_labels, pred_labels, labels=SORTED_LABEL_IDS, average=None, zero_division=0)

    for pos, class_id in enumerate(SORTED_LABEL_IDS):
        dialect = ID2LABEL[class_id]
        results_text += f"\n{dialect}:"
        results_text += f"\n  Precision: {precision_per_class[pos]:.4f}"
        results_text += f"\n  Recall:    {recall_per_class[pos]:.4f}"
        results_text += f"\n  F1-Score:  {f1_per_class[pos]:.4f}"
        results_text += f"\n  Support:   {support[pos]}"
        results_text += "\n"

    txt_filename = f'results_{model_name}_seed{effective_seed}.txt'
    with open(txt_filename, 'w', encoding='utf-8') as f:
        f.write(results_text)
    print(f"Results saved to: {txt_filename}")
    print(results_text)

    # ============================================================
    # SAVE CONFUSION MATRIX AS IMAGE
    # ============================================================
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=SORTED_LABEL_NAMES,
                yticklabels=SORTED_LABEL_NAMES,
                cbar_kws={'label': 'Count'})
    plt.title(f'Confusion Matrix - {model_name} (seed={effective_seed})', fontsize=16, fontweight='bold')
    plt.ylabel('True Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.tight_layout()

    plot_filename = f'confusion_matrix_{model_name}_seed{effective_seed}.png'
    plt.savefig(plot_filename, dpi=300, bbox_inches='tight')
    print(f"Confusion matrix saved to: {plot_filename}")
    plt.show()

    # ============================================================
    # SAVE RESULTS AS JSON
    # ============================================================
    results = {
        'model_name': model_name,
        'model_id': model_id,
        'model_save_path': model_save_path,
        'seed': effective_seed,
        'training_date': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),
        'accuracy': float(accuracy),
        'precision': float(precision_m),
        'recall': float(recall_m),
        'f1': float(f1_m),
        'precision_macro': float(precision_m),
        'recall_macro': float(recall_m),
        'f1_macro': float(f1_m),
        'precision_weighted': float(precision_w),
        'recall_weighted': float(recall_w),
        'f1_weighted': float(f1_w),
        'per_class_metrics': {
            ID2LABEL[class_id]: {
                'precision': float(precision_per_class[pos]),
                'recall': float(recall_per_class[pos]),
                'f1': float(f1_per_class[pos]),
                'support': int(support[pos])
            }
            for pos, class_id in enumerate(SORTED_LABEL_IDS)
        },
        'confusion_matrix': cm.tolist(),
        'confusion_matrix_label_order': SORTED_LABEL_NAMES,

        'true_labels': [int(x) for x in true_labels],
        'pred_labels': [int(x) for x in pred_labels],
        'training_config': {
            'batch_size': BATCH_SIZE,
            'learning_rate': LEARNING_RATE,
            'num_epochs': NUM_EPOCHS,
            'max_length': MAX_LENGTH,
            'use_lora': USE_LORA,
            'use_8bit': USE_8BIT,
            'seed': effective_seed
        }
    }

    json_filename = f'results_{model_name}_seed{effective_seed}.json'
    with open(json_filename, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    print(f"JSON results saved to: {json_filename}")

    # ============================================================
    # SAVE TRAINING HISTORY PLOT
    # ============================================================
    try:
        log_history = trainer.state.log_history

        train_loss = [log['loss'] for log in log_history if 'loss' in log]
        eval_loss = [log['eval_loss'] for log in log_history if 'eval_loss' in log]
        eval_f1 = [log['eval_f1'] for log in log_history if 'eval_f1' in log]

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        axes[0].plot(train_loss, label='Training Loss', marker='o')
        if eval_loss:
            axes[0].plot(eval_loss, label='Validation Loss', marker='s')
        axes[0].set_xlabel('Step/Epoch')
        axes[0].set_ylabel('Loss')
        axes[0].set_title(f'Training/Validation Loss - {model_name} (seed={effective_seed})')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)

        if eval_f1:
            axes[1].plot(eval_f1, label='Validation F1 (macro)', marker='s', color='green')
            axes[1].set_xlabel('Epoch')
            axes[1].set_ylabel('F1 Score')
            axes[1].set_title(f'Validation F1 Score - {model_name} (seed={effective_seed})')
            axes[1].legend()
            axes[1].grid(True, alpha=0.3)

        plt.tight_layout()

        training_plot_filename = f'training_curves_{model_name}_seed{effective_seed}.png'
        plt.savefig(training_plot_filename, dpi=300, bbox_inches='tight')
        print(f"Training curves saved to: {training_plot_filename}")
        plt.show()

    except Exception as e:
        print(f"⚠ Could not save training curves: {e}")

    print(f"\n{'='*60}")
    print(f"ALL RESULTS SAVED FOR: {model_name}  (seed={effective_seed})")
    print(f"{'='*60}")
    print(f"Model:           {model_save_path}")
    print(f"Text report:     {txt_filename}")
    print(f"Confusion matrix: {plot_filename}")
    print(f"JSON data:       {json_filename}")
    if 'training_plot_filename' in locals():
        print(f"Training curves: {training_plot_filename}")
    print(f"{'='*60}\n")

    del model
    torch.cuda.empty_cache()

    return trainer, results

print("Fine-tuning function ready")


##2.3 Multi-Seed Fune-Tuned Runs + Significance Testing


**Expect this to take a while**, especially `allam-7b` and `qwen2.5-7b`
(LoRA + 8-bit)


In [ ]:
# ==============================================================================================
# Multi-Seed Runs for marbert', 'arabert-v2', 'camelbert-da', 'qarib', 'allam-7b', 'qwen2.5-7b'
# =============================================================================================
import os

MULTISEED_MODELS = ['marbert', 'arabert-v2', 'camelbert-da', 'qarib', 'allam-7b', 'qwen2.5-7b']
multiseed_results = {}

for model_name in MULTISEED_MODELS:
    multiseed_results[model_name] = {}
    for seed in SEEDS:
        result_path = f'results_{model_name}_seed{seed}.json'

        if os.path.exists(result_path):
            print(f" Found existing result, skipping: {result_path}")
            with open(result_path) as f:
                multiseed_results[model_name][seed] = json.load(f)
            continue

        print(f"\n{'*'*70}\nRunning {model_name} with seed={seed}\n{'*'*70}")
        try:
            _, results = finetune_model(
                model_name, FINETUNING_MODELS[model_name], dataset_dict, seed=seed
            )
            multiseed_results[model_name][seed] = results
        except Exception as e:
            print(f"⚠ {model_name} (seed={seed}) failed: {e}")
            multiseed_results[model_name][seed] = None

        torch.cuda.empty_cache()

# =====================================================
# Aggregate: mean ± std across seeds
# =====================================================

summary_rows = []
for model_name in MULTISEED_MODELS:
    accs, f1s = [], []
    for seed in SEEDS:
        r = multiseed_results[model_name].get(seed)
        if r is not None:
            accs.append(r['accuracy'])
            f1s.append(r.get('f1_macro', r.get('f1')))

    if accs:
        summary_rows.append({
            'model': model_name,
            'n_seeds': len(accs),
            'accuracy_mean': float(np.mean(accs)),
            'accuracy_std': float(np.std(accs, ddof=1)) if len(accs) > 1 else 0.0,
            'f1_macro_mean': float(np.mean(f1s)),
            'f1_macro_std': float(np.std(f1s, ddof=1)) if len(f1s) > 1 else 0.0,
        })

print(f"\n{'Model':<15} {'Seeds':<7} {'Accuracy (mean ± sd)':<24} {'Macro F1 (mean ± sd)'}")
print("-" * 75)
for row in summary_rows:
    print(f"{row['model']:<15} {row['n_seeds']:<7} "
          f"{row['accuracy_mean']*100:>6.2f}% ± {row['accuracy_std']*100:.2f}pp      "
          f"{row['f1_macro_mean']*100:>6.2f}% ± {row['f1_macro_std']*100:.2f}pp")

with open('results_multiseed_summary.json', 'w') as f:
    json.dump(summary_rows, f, indent=2)
print("\n Summary saved to: results_multiseed_summary.json")


In [ ]:
# =====================================================
# Significance Testing: McNemar's test + Wilson 95% CIs
# =====================================================
!pip install -q statsmodels

PRIMARY_SEED = 42
comparison_models = ['qarib', 'marbert', 'arabert-v2', 'camelbert-da', 'allam-7b', 'qwen2.5-7b']

preds = {}
true_labels_ref = None
significance_results = {
    'seed': PRIMARY_SEED,
    'test_set_size': None,
    'wilson_ci': {},
    'mcnemar_vs_qarib': {},
    'mcnemar_7b_pairwise': {},
}
for m in comparison_models:
    path = f'results_{m}_seed{PRIMARY_SEED}.json'
    if not os.path.exists(path):
        print(f"⚠ Missing {path}")
        continue
    with open(path) as f:
        r = json.load(f)
    if 'true_labels' not in r or 'pred_labels' not in r:
        print(f"⚠ {path} has no saved per-sample predictions ")
        continue
    preds[m] = np.array(r['pred_labels'])
    true_labels_ref = np.array(r['true_labels'])

print("="*60)
print("95% Wilson Confidence Intervals (accuracy)")
print("="*60)


if preds:
    ci_input = {m: (int(np.sum(preds[m] == true_labels_ref)), len(true_labels_ref)) for m in preds}
    print_wilson_ci_table(ci_input)
    significance_results['test_set_size'] = len(true_labels_ref)
    for m, (n_correct, n_total) in ci_input.items():
        point, low, high = wilson_ci(n_correct, n_total)
        significance_results['wilson_ci'][m] = {
            'accuracy': point, 'ci_low': low, 'ci_high': high,
            'n_correct': n_correct, 'n_total': n_total
        }
else:
    print("No predictions loaded -- nothing to test yet.")

print("\n" + "="*60)
print("Pairwise McNemar's tests vs. QARIB")
print("="*60)
if 'qarib' in preds:
    for m in preds:
        if m == 'qarib':
            continue
        pval, table = mcnemar_test(true_labels_ref, preds['qarib'], preds[m], 'QARIB', m)
        significance_results['mcnemar_vs_qarib'][m] = {'pvalue': pval, 'contingency_table': table}
        print()
else:
    print("QARIB predictions not found -- cannot run comparisons against QARIB.")

print("="*60)
print("Pairwise McNemar's tests among the three 7B models")
print("(Qwen2.5-7B vs ALLaM-7B vs CAMeLBERT-DA -- checking whether their")
print(" close accuracies are actually distinguishable)")
print("="*60)
sevenb_models = [m for m in ['qwen2.5-7b', 'allam-7b', 'camelbert-da'] if m in preds]
for i in range(len(sevenb_models)):
    for j in range(i + 1, len(sevenb_models)):
        a, b = sevenb_models[i], sevenb_models[j]
        pval, table = mcnemar_test(true_labels_ref, preds[a], preds[b], a, b)
        significance_results['mcnemar_7b_pairwise'][f'{a}_vs_{b}'] = {'pvalue': pval, 'contingency_table': table}
        print()
sig_filename = f'results_significance_testing_seed{PRIMARY_SEED}.json'
with open(sig_filename, 'w', encoding='utf-8') as f:
    json.dump(significance_results, f, indent=2, ensure_ascii=False)
print(f"\n Significance testing results saved to: {sig_filename}")

In [ ]:
# ===========================================================
# Seed-Level Statistical Comparison (mean ± SD across seeds)
# ===========================================================

from scipy import stats
import numpy as np, json, os, glob

def load_seed_values(model_name, metric='f1_macro'):
    """Load {seed: metric_value} for a model from all completed
    results_<model>_seed<seed>.json files."""
    values = {}
    for fpath in sorted(glob.glob(f'results_{model_name}_seed*.json')):
        with open(fpath) as f:
            data = json.load(f)
        seed = data.get('seed')
        if seed is None:
            seed = int(fpath.split('seed')[-1].replace('.json', ''))
        val = data.get(metric, data.get('f1_macro', data.get('f1')) if metric == 'f1_macro' else data.get('accuracy'))
        if val is not None:
            values[seed] = val
    return values

def paired_seed_comparison(model_a, model_b, metric='f1_macro'):
    """Paired t-test (and Wilcoxon signed-rank if enough pairs) comparing
    two models' per-seed metric values, paired by shared seed.
    Returns a results dict (or None if too few shared seeds)."""
    vals_a = load_seed_values(model_a, metric)
    vals_b = load_seed_values(model_b, metric)
    common_seeds = sorted(set(vals_a) & set(vals_b))

    print(f"\n{model_a} vs {model_b}  ({metric})")
    if len(common_seeds) < 2:
        print(f"  ⚠ only {len(common_seeds)} shared seed(s) found — need at least 2 for a paired test.")
        return None

    a = np.array([vals_a[s] for s in common_seeds])
    b = np.array([vals_b[s] for s in common_seeds])

    print(f"  n = {len(common_seeds)} shared seeds: {common_seeds}")
    print(f"  {model_a}: {a.mean()*100:.2f}% ± {a.std(ddof=1)*100:.2f}pp  "
          f"(per-seed: {[f'{v*100:.2f}' for v in a]})")
    print(f"  {model_b}: {b.mean()*100:.2f}% ± {b.std(ddof=1)*100:.2f}pp  "
          f"(per-seed: {[f'{v*100:.2f}' for v in b]})")

    t_stat, t_p = stats.ttest_rel(a, b)
    verdict = "significant at p<0.05" if t_p < 0.05 else "NOT significant at p<0.05"
    print(f"  Paired t-test:        t={t_stat:.3f}, p={t_p:.4f}  ({verdict})")

    result = {
        'model_a': model_a, 'model_b': model_b, 'metric': metric,
        'common_seeds': common_seeds, 'n': len(common_seeds),
        'mean_a': float(a.mean()), 'sd_a': float(a.std(ddof=1)),
        'mean_b': float(b.mean()), 'sd_b': float(b.std(ddof=1)),
        'values_a': a.tolist(), 'values_b': b.tolist(),
        't_stat': float(t_stat), 't_pvalue': float(t_p),
    }

    if len(common_seeds) >= 5:
        w_stat, w_p = stats.wilcoxon(a, b)
        print(f" Wilcoxon signed-rank: W={w_stat:.3f}, p={w_p:.4f}")
        result['wilcoxon_stat'] = float(w_stat)
        result['wilcoxon_pvalue'] = float(w_p)
    else:
        print(f"  (Wilcoxon signed-rank skipped — needs more pairs than n={len(common_seeds)} to be meaningful)")

    return result
# ---- Run all pairwise comparisons across the six models ----
MULTISEED_MODELS = ['marbert', 'arabert-v2', 'camelbert-da', 'qarib', 'allam-7b', 'qwen2.5-7b']
METRIC = 'f1_macro'
seed_test_results = []
for i in range(len(MULTISEED_MODELS)):
    for j in range(i + 1, len(MULTISEED_MODELS)):
        r = paired_seed_comparison(MULTISEED_MODELS[i], MULTISEED_MODELS[j], metric=METRIC)
        if r is not None:
            seed_test_results.append(r)

# Bonferroni correction across however many pairs were actually tested
n_tests = len(seed_test_results)
bonferroni_alpha = 0.05 / n_tests if n_tests > 0 else None
print(f"\n{'='*60}")
print(f"Bonferroni-corrected significance (n={n_tests} tests, α=0.05/{n_tests}={bonferroni_alpha:.4f})" if n_tests else "No valid comparisons to correct.")
print(f"{'='*60}")
for r in seed_test_results:
    r['bonferroni_alpha'] = bonferroni_alpha
    r['significant_bonferroni'] = bool(r['t_pvalue'] < bonferroni_alpha) if bonferroni_alpha else None
    if r['significant_bonferroni']:
        print(f"  {r['model_a']} vs {r['model_b']}: p={r['t_pvalue']:.4f}  -- survives correction")

# ---- Save everything ----
seed_test_filename = f'results_seedlevel_ttest_{METRIC}.json'
with open(seed_test_filename, 'w', encoding='utf-8') as f:
    json.dump(seed_test_results, f, indent=2, ensure_ascii=False)
print(f"\n Seed-level test results saved to: {seed_test_filename}")


#3. Binary experiment


In [ ]:
# =====================================================
# Binary Classification Configuration
# =====================================================

BINARY_PAIRS = {
    'hijazi_hasawi': {
        'dialects': ['Hijazi', 'Hasawi'],
        'original_labels': [LABEL2ID['Hijazi'], LABEL2ID['Hasawi']],
        'label2id': {'Hijazi': 0, 'Hasawi': 1},
        'id2label': {0: 'Hijazi', 1: 'Hasawi'},
        'description': 'Maximal geographic distance (West vs. East)',
        'baseline': 64.5  # From Alqurashi 2022
    },
    'hijazi_najdi': {
        'dialects': ['Hijazi', 'Najdi'],
        'original_labels': [LABEL2ID['Hijazi'], LABEL2ID['Najdi']],
        'label2id': {'Hijazi': 0, 'Najdi': 1},
        'id2label': {0: 'Hijazi', 1: 'Najdi'},
        'description': 'Adjacent regions (West vs. Central)',
        'baseline': None
    },
    'najdi_hasawi': {
        'dialects': ['Najdi', 'Hasawi'],
        'original_labels': [LABEL2ID['Najdi'], LABEL2ID['Hasawi']],
        'label2id': {'Najdi': 0, 'Hasawi': 1},
        'id2label': {0: 'Najdi', 1: 'Hasawi'},
        'description': 'Medium distance (Central vs. East)',
        'baseline': None
    }
}

# Model configuration (using QARIB - best model)
BINARY_MODEL = {
    'name': 'qarib',
    'id': 'ahmedabdelali/bert-base-qarib',
    'type': 'bert'
}

# Training parameters (same as the main experiments)
BINARY_BATCH_SIZE = 8
BINARY_EPOCHS = 5
BINARY_LEARNING_RATE = 2e-5

print("Binary classification configuration loaded")
print(f"\nWill run 3 experiments:")
for pair_name, config in BINARY_PAIRS.items():
    print(f"  - {config['dialects'][0]} vs {config['dialects'][1]} "
          f"(true label ids {config['original_labels']})")


In [ ]:
# =====================================================
# Filter Dataset for Binary Task
# =====================================================

def create_binary_dataset(train_df, val_df, test_df, pair_config):
    """
    Filter dataset for a specific dialect pair and create binary labels

    Args:
        train_df, val_df, test_df: Your original dataframes
        pair_config: Configuration for the dialect pair

    Returns:
        Binary datasets (train, val, test)
    """
    dialect1, dialect2 = pair_config['dialects']
    original_labels = pair_config['original_labels']
    label2id = pair_config['label2id']

    print(f"\n{'='*60}")
    print(f"Creating binary dataset: {dialect1} vs {dialect2}")
    print(f"Description: {pair_config['description']}")
    print(f"{'='*60}")

    def filter_and_relabel(df, split_name):
        """Filter for two dialects and create binary labels"""
        # Filter for the two dialects only
        df_filtered = df[df['label'].isin(original_labels)].copy()

        # Map to binary labels (0 or 1)
        df_filtered['label'] = df_filtered['label'].map({
            original_labels[0]: 0,  # First dialect → 0
            original_labels[1]: 1   # Second dialect → 1
        })

        df_filtered = df_filtered.reset_index(drop=True)

        # Count samples
        n_dialect1 = (df_filtered['label'] == 0).sum()
        n_dialect2 = (df_filtered['label'] == 1).sum()

        print(f"  {split_name:12s}: {len(df_filtered):4d} samples "
              f"({dialect1}: {n_dialect1}, {dialect2}: {n_dialect2})")

        return df_filtered

    # Filter all splits
    binary_train = filter_and_relabel(train_df, 'Train')
    binary_val = filter_and_relabel(val_df, 'Validation')
    binary_test = filter_and_relabel(test_df, 'Test')

    # Convert to HuggingFace datasets
    train_dataset = Dataset.from_pandas(binary_train[['text', 'label']], preserve_index=False)
    val_dataset = Dataset.from_pandas(binary_val[['text', 'label']], preserve_index=False)
    test_dataset = Dataset.from_pandas(binary_test[['text', 'label']], preserve_index=False)

    dataset_dict = DatasetDict({
        'train': train_dataset,
        'validation': val_dataset,
        'test': test_dataset
    })

    return dataset_dict

print("Binary dataset creation function loaded")


In [ ]:
# =====================================================
# Binary Fine-Tuning Function
# =====================================================

def finetune_binary_qarib(pair_name, pair_config, train_df, val_df, test_df):
    """
    Fine-tune QARIB on a binary classification task

    Args:
        pair_name: Name of the dialect pair (e.g., 'hijazi_hasawi')
        pair_config: Configuration for this pair
        train_df, val_df, test_df: Original dataframes

    Returns:
        Results dictionary
    """
    import datetime
    from datetime import datetime as dt

    dialect1, dialect2 = pair_config['dialects']

    print(f"\n{'#'*70}")
    print(f"# BINARY CLASSIFICATION: {dialect1} vs {dialect2}")
    print(f"{'#'*70}")
    print(f"Model: {BINARY_MODEL['id']}")
    print(f"Description: {pair_config['description']}")
    if pair_config['baseline']:
        print(f"Baseline to beat: {pair_config['baseline']}%")
    print(f"{'#'*70}\n")

    # 1. Create binary dataset
    dataset_dict = create_binary_dataset(train_df, val_df, test_df, pair_config)

    # 2. Load tokenizer
    print(f"\nLoading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(BINARY_MODEL['id'])

    # 3. Tokenize
    print("Tokenizing dataset...")
    def tokenize_function(examples):
        return tokenizer(
            examples['text'],
            padding='max_length',
            truncation=True,
            max_length=MAX_LENGTH
        )

    tokenized_data = dataset_dict.map(tokenize_function, batched=True)

    # 4. Load model
    print(f"\nLoading model for binary classification...")
    model = AutoModelForSequenceClassification.from_pretrained(
        BINARY_MODEL['id'],
        num_labels=2,
        id2label=pair_config['id2label'],
        label2id=pair_config['label2id']
    )
    model.to(device)

    # 5. Training arguments
    output_dir = f"./binary_models/{pair_name}"
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=BINARY_EPOCHS,
        per_device_train_batch_size=BINARY_BATCH_SIZE,
        per_device_eval_batch_size=BINARY_BATCH_SIZE,
        learning_rate=BINARY_LEARNING_RATE,
        weight_decay=0.01,
        warmup_steps=100,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        logging_steps="epoch",
        seed=42,
        fp16=True,
        report_to="none"
    )

    # 6. Compute metrics
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        predictions = np.argmax(logits, axis=-1)

        accuracy = accuracy_score(labels, predictions)
        precision, recall, f1, _ = precision_recall_fscore_support(
            labels, predictions, average='weighted', zero_division=0
        )

        return {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1
        }

    # 7. Create trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_data['train'],
        eval_dataset=tokenized_data['validation'],
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
    )

    # 8. Train
    print("\nStarting training...")
    train_result = trainer.train()

    # 9. Evaluate on test set
    print("\nEvaluating on test set...")
    test_results = trainer.evaluate(tokenized_data['test'])

    # 10. Get predictions for confusion matrix
    predictions = trainer.predict(tokenized_data['test'])
    y_pred = np.argmax(predictions.predictions, axis=-1)
    y_true = predictions.label_ids

    # 11. Print results
    print(f"\n{'='*70}")
    print(f"BINARY RESULTS: {dialect1} vs {dialect2}")
    print(f"{'='*70}")
    print(f"\nOVERALL METRICS:")
    print(f"Accuracy:  {test_results['eval_accuracy']:.4f} ({test_results['eval_accuracy']*100:.2f}%)")
    print(f"Precision: {test_results['eval_precision']:.4f}")
    print(f"Recall:    {test_results['eval_recall']:.4f}")
    print(f"F1 Score:  {test_results['eval_f1']:.4f}")
    print(f"{'='*70}\n")

    # 12. Detailed classification report
    print("DETAILED CLASSIFICATION REPORT:")
    report_str = classification_report(
        y_true,
        y_pred,
        target_names=[dialect1, dialect2],
        digits=4
    )
    print(report_str)

    # 13. Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    print("\nCONFUSION MATRIX:")
    print(f"              Predicted")
    print(f"              {dialect1:<12s} {dialect2:<12s}")
    print(f"Actual {dialect1:<12s} {cm[0,0]:<12d} {cm[0,1]:<12d}")
    print(f"       {dialect2:<12s} {cm[1,0]:<12d} {cm[1,1]:<12d}")


    # 15. Package results
    results_dict = {
        'pair_name': pair_name,
        'dialects': pair_config['dialects'],
        'description': pair_config['description'],
        'model': BINARY_MODEL['id'],
        'date': dt.now().strftime("%Y-%m-%d %H:%M:%S"),
        'metrics': {
            'accuracy': float(test_results['eval_accuracy']),
            'precision': float(test_results['eval_precision']),
            'recall': float(test_results['eval_recall']),
            'f1': float(test_results['eval_f1'])
        },
        'baseline': pair_config['baseline'],
        'confusion_matrix': cm.tolist(),
        'classification_report_dict': classification_report(
            y_true, y_pred,
            target_names=[dialect1, dialect2],
            digits=4,
            output_dict=True
        ),
        'classification_report_text': report_str
    }

    # 16. Save results
    import os
    os.makedirs('./results', exist_ok=True)

    # Save JSON
    json_file = f"./results/binary_{pair_name}_qarib.json"
    with open(json_file, 'w', encoding='utf-8') as f:
        json.dump(results_dict, f, indent=2, ensure_ascii=False)
    print(f"\n Results saved to: {json_file}")

    # Save text report
    txt_file = f"./results/binary_{pair_name}_qarib.txt"
    with open(txt_file, 'w', encoding='utf-8') as f:
        f.write(f"{'='*70}\n")
        f.write(f"BINARY CLASSIFICATION RESULTS: {dialect1} vs {dialect2}\n")
        f.write(f"{'='*70}\n")
        f.write(f"Model: {BINARY_MODEL['id']}\n")
        f.write(f"Date: {results_dict['date']}\n")
        f.write(f"Description: {pair_config['description']}\n")

        f.write(f"OVERALL METRICS:\n")
        f.write(f"Accuracy:  {test_results['eval_accuracy']:.4f} ({test_results['eval_accuracy']*100:.2f}%)\n")
        f.write(f"Precision: {test_results['eval_precision']:.4f}\n")
        f.write(f"Recall:    {test_results['eval_recall']:.4f}\n")
        f.write(f"F1 Score:  {test_results['eval_f1']:.4f}\n")
        f.write(f"{'='*70}\n\n")

        f.write("DETAILED CLASSIFICATION REPORT:\n")
        f.write(report_str)

        f.write(f"\n\nCONFUSION MATRIX:\n")
        f.write(str(cm))

        if pair_config['baseline']:
            diff = (test_results['eval_accuracy'] * 100) - pair_config['baseline']
            f.write(f"\n\nCOMPARISON TO BASELINE:\n")
            f.write(f"Alqurashi 2022: {pair_config['baseline']:.2f}%\n")
            f.write(f"QARIB (ours):   {test_results['eval_accuracy']*100:.2f}%\n")

    print(f"Text report saved to: {txt_file}")

    # Clear GPU memory
    del model
    del trainer
    torch.cuda.empty_cache()

    return results_dict

print("Binary fine-tuning function loaded")



In [ ]:
# =====================================================
# Binary Experiments
# =====================================================

print("\n" + "="*70)
print("STARTING BINARY CLASSIFICATION EXPERIMENTS")
print("="*70)

# Store all results
all_binary_results = {}

# Run each experiment
for i, (pair_name, pair_config) in enumerate(BINARY_PAIRS.items(), 1):
    print(f"\n\n{'#'*70}")
    print(f"# EXPERIMENT {i}/3: {pair_name.upper().replace('_', ' ')}")
    print(f"{'#'*70}")

    try:
        results = finetune_binary_qarib(
            pair_name,
            pair_config,
            train_df,
            val_df,
            test_df
        )
        all_binary_results[pair_name] = results
        print(f"\n Experiment {i}/3 complete!")

    except Exception as e:
        print(f"\n✗ Error in {pair_name}: {str(e)}")
        import traceback
        traceback.print_exc()
        all_binary_results[pair_name] = {'error': str(e)}

# Print summary
print("\n\n" + "="*70)
print("BINARY CLASSIFICATION SUMMARY")
print("="*70)
print(f"\n{'Dialect Pair':<25} {'Accuracy':<12} {'F1 Score':<12} {'vs Baseline'}")
print("-"*70)

for pair_name, results in all_binary_results.items():
    if 'error' not in results:
        pair_config = BINARY_PAIRS[pair_name]
        dialects = " vs ".join(pair_config['dialects'])
        acc = results['metrics']['accuracy']
        f1 = results['metrics']['f1']

        baseline_str = ""
        if pair_config['baseline']:
            diff = (acc * 100) - pair_config['baseline']
            baseline_str = f"{diff:+.1f}pp"

        print(f"{dialects:<25} {acc*100:>6.2f}%     {f1*100:>6.2f}%     {baseline_str}")

print("="*70)
print("\n All binary experiments complete!")
print("\nResults saved in:")
print("  - JSON files: ./results/binary_*.json")
print("  - Text reports: ./results/binary_*.txt")



#3.  3-Way Classification Experiment

In [ ]:
# =====================================================
# 3-WAY CLASSIFICATION: Leave-One-Dialect-Out
# =====================================================

def create_leave_one_out_dataset(train_df, val_df, test_df, exclude_dialect):

    print(f"\nCreating 3-way dataset (excluding {exclude_dialect})...")

    EXCLUDE_LABEL = LABEL2ID[exclude_dialect]
    KEEP_LABELS = [i for i in SORTED_LABEL_IDS if i != EXCLUDE_LABEL]

    remap = {orig_id: new_id for new_id, orig_id in enumerate(KEEP_LABELS)}
    id2label_3way = {new_id: ID2LABEL[orig_id] for orig_id, new_id in remap.items()}
    label2id_3way = {v: k for k, v in id2label_3way.items()}

    def filter_and_remap(df, split_name):
        filtered = df[df['label'].isin(KEEP_LABELS)].copy()
        filtered['label'] = filtered['label'].map(remap)
        filtered = filtered.reset_index(drop=True)
        print(f"  {split_name:12s}: {len(filtered):4d} samples (was {len(df)})")
        return filtered

    train_3way = filter_and_remap(train_df, 'Train')
    val_3way = filter_and_remap(val_df, 'Validation')
    test_3way = filter_and_remap(test_df, 'Test')

    print(f"  3-way label mapping (local id -> dialect): {id2label_3way}")

    train_dataset = Dataset.from_pandas(train_3way[['text', 'label']], preserve_index=False)
    val_dataset = Dataset.from_pandas(val_3way[['text', 'label']], preserve_index=False)
    test_dataset = Dataset.from_pandas(test_3way[['text', 'label']], preserve_index=False)

    dataset_dict = DatasetDict({'train': train_dataset, 'validation': val_dataset, 'test': test_dataset})
    return dataset_dict, id2label_3way, label2id_3way


def compute_metrics_3way(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = accuracy_score(labels, predictions)
    precision_w, recall_w, f1_w, _ = precision_recall_fscore_support(labels, predictions, average='weighted', zero_division=0)
    precision_m, recall_m, f1_m, _ = precision_recall_fscore_support(labels, predictions, average='macro', zero_division=0)
    return {
        'accuracy': accuracy, 'precision': precision_m, 'recall': recall_m, 'f1': f1_m,
        'precision_weighted': precision_w, 'recall_weighted': recall_w, 'f1_weighted': f1_w,
    }


def run_leave_one_out_3way(exclude_dialect, four_way_baseline_accuracy, seed=42):

    tag = exclude_dialect.lower()
    json_path = f'./results/3way_exclude{exclude_dialect}_qarib_seed{seed}.json'
    txt_path = f'./results/3way_exclude{exclude_dialect}_qarib_seed{seed}.txt'

    if os.path.exists(json_path):
        print(f"Found existing result, skipping: {json_path}")
        with open(json_path) as f:
            return json.load(f)

    print("\n" + "="*70)
    print(f"3-WAY CLASSIFICATION EXPERIMENT -- excluding {exclude_dialect}")
    print("="*70)

    three_way_data, id2label_3way, label2id_3way = create_leave_one_out_dataset(
        train_df, val_df, test_df, exclude_dialect
    )
    sorted_3way_ids = sorted(id2label_3way)
    sorted_3way_names = [id2label_3way[i] for i in sorted_3way_ids]

    tokenizer = AutoTokenizer.from_pretrained('ahmedabdelali/bert-base-qarib')

    def tokenize_function(examples):
        return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=MAX_LENGTH)

    tokenized_3way = three_way_data.map(tokenize_function, batched=True)

    model_3way = AutoModelForSequenceClassification.from_pretrained(
        'ahmedabdelali/bert-base-qarib', num_labels=3, id2label=id2label_3way, label2id=label2id_3way
    )
    model_3way.to(device)

    training_args_3way = TrainingArguments(
        output_dir=f"./3way_exclude{exclude_dialect}_model",
        num_train_epochs=5, per_device_train_batch_size=8, per_device_eval_batch_size=8,
        learning_rate=2e-5, weight_decay=0.01, warmup_steps=100,
        eval_strategy="epoch", save_strategy="epoch",
        load_best_model_at_end=True, metric_for_best_model="f1", greater_is_better=True,
        logging_steps="epoch", seed=seed, fp16=torch.cuda.is_available(), report_to="none"
    )

    trainer_3way = Trainer(
        model=model_3way, args=training_args_3way,
        train_dataset=tokenized_3way['train'], eval_dataset=tokenized_3way['validation'],
        compute_metrics=compute_metrics_3way,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
    )

    print(f"\nStarting 3-way training (excluding {exclude_dialect})...")
    trainer_3way.train()

    results_3way = trainer_3way.evaluate(tokenized_3way['test'])
    predictions_3way = trainer_3way.predict(tokenized_3way['test'])
    y_pred_3way = np.argmax(predictions_3way.predictions, axis=-1)
    y_true_3way = predictions_3way.label_ids

    acc_pct = results_3way['eval_accuracy'] * 100

    print("\n" + "="*70)
    print(f"3-WAY RESULTS (excluding {exclude_dialect})")
    print("="*70)
    print(f"Accuracy:    {results_3way['eval_accuracy']:.4f} ({acc_pct:.2f}%)")
    print(f"Macro F1:    {results_3way['eval_f1']:.4f}")
    print(f"Weighted F1: {results_3way['eval_f1_weighted']:.4f}")
    print(f"\n4-way baseline (as provided): {four_way_baseline_accuracy:.2f}%")


    report_text = classification_report(y_true_3way, y_pred_3way, labels=sorted_3way_ids,
                                          target_names=sorted_3way_names, digits=4)
    print("\n" + report_text)

    os.makedirs('./results', exist_ok=True)
    with open(txt_path, 'w') as f:
        f.write("="*70 + f"\n3-WAY CLASSIFICATION (Excluding {exclude_dialect})\n" + "="*70 + "\n")
        f.write(f"Accuracy:    {results_3way['eval_accuracy']:.4f}\n")
        f.write(f"Macro F1:    {results_3way['eval_f1']:.4f}\n")
        f.write(f"Weighted F1: {results_3way['eval_f1_weighted']:.4f}\n\n")
        f.write("COMPARISON:\n")
        f.write(f"  4-way baseline: {four_way_baseline_accuracy:.2f}%\n")
        f.write(f"  3-way:          {acc_pct:.2f}%\n")
        f.write(report_text)

    result_dict = {
        'accuracy': float(results_3way['eval_accuracy']),
        'f1_macro': float(results_3way['eval_f1']),
        'f1_weighted': float(results_3way['eval_f1_weighted']),
        'excluded_dialect': exclude_dialect,
        'kept_dialects': sorted_3way_names,
        'label_mapping': id2label_3way,
        'four_way_baseline_accuracy': four_way_baseline_accuracy,
        'seed': seed,
        'true_labels': [int(x) for x in y_true_3way],
        'pred_labels': [int(x) for x in y_pred_3way],
    }
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(result_dict, f, indent=2, ensure_ascii=False)

    print(f"\n✓ Results saved to: {txt_path}")
    print(f"✓ Results saved to: {json_path}")

    del model_3way, tokenizer, trainer_3way
    gc.collect()
    torch.cuda.empty_cache()

    return result_dict

# ---- Run for Janobi and Hasawi, both against the SAME,
#      selected seed=42 4-way baseline ----
FOUR_WAY_BASELINE_ACCURACY_SEED42 = 43.77  # QARIB, seed=42, macro-F1-selected (see Table 1)

result_exclude_janobi = run_leave_one_out_3way('Janobi', FOUR_WAY_BASELINE_ACCURACY_SEED42, seed=42)
result_exclude_hasawi = run_leave_one_out_3way('Hasawi', FOUR_WAY_BASELINE_ACCURACY_SEED42, seed=42)

print("\n" + "="*70)
print("LEAVE-ONE-OUT COMPARISON (vs. 4-way baseline "
      f"{FOUR_WAY_BASELINE_ACCURACY_SEED42:.2f}%)")
print("="*70)
for r in [result_exclude_janobi, result_exclude_hasawi]:
    print(f"  Exclude {r['excluded_dialect']:<8s}: {r['accuracy']*100:.2f}% accuracy  ")
